# (Homework) Week 7 - DataScience Bootcamp Fall 2025

All solution cells are replaced with `# TODO` placeholders so you can fill them in.

**Name:** Advait Jishnani\
**Email:** AJ4700@nyu.edu

---

## Problem A: Bayesian Dice Game (Posterior Inference)

You are playing a dice game at a carnival. The operator has **three dice**, each with different biases for rolling a six:

| Die | P(6) | P(1–5) |
|-----|------|--------|
| A   | 0.10 | 0.90   |
| B   | 0.30 | 0.70   |
| C   | 0.60 | 0.40   |

Before each round, the operator secretly picks one die at random (each equally likely). He then rolls it **10 times** and tells you how many sixes appeared.

Your job is to infer which die was used using **Bayes’ Theorem**:

$$ P(Die|k) = \frac{P(k|Die)P(Die)}{\sum_{d} P(k|d)P(d)} $$

where $P(k|Die)$ follows a Binomial (n=10, p_i) distribution.

**Tasks:**
1. Simulate the experiment by picking a random die and rolling it 10 times.
2. Compute posterior probabilities for each die given observed sixes.
3. Plot likelihoods and posterior probabilities.
4. Evaluate inference accuracy over 100 rounds.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

# Dice setup
dice_probs = {'A': 0.1, 'B': 0.3, 'C': 0.6}
dice_names = list(dice_probs.keys())
prior = {die: 1/3 for die in dice_names}
n_rolls = 10

# Compute binomial probability mass function
def binomial_prob(n, k, p):
    """Compute P(X = k) for Binomial(n, p)"""
    return math.comb(n, k) * (p**k) * ((1-p)**(n-k))

def simulate_round():
    """Randomly pick a die and roll it 10 times, return (die, num_sixes)"""
    true_die = np.random.choice(dice_names)
    p = dice_probs[true_die]
    k = np.random.binomial(n_rolls, p)
    return true_die, k

def posterior_given_k(k):
    """Compute posterior probabilities for each die given k sixes observed"""
    likelihoods = {die: binomial_prob(n_rolls, k, dice_probs[die]) for die in dice_names}
    
    # Bayes' theorem: P(die|k) = P(k|die) * P(die) / P(k)
    numerator = {die: likelihoods[die] * prior[die] for die in dice_names}
    denominator = sum(numerator.values())
    
    posterior = {die: numerator[die] / denominator for die in dice_names}
    return [posterior[die] for die in dice_names]

# Example run
true_die, k = simulate_round()
posterior = posterior_given_k(k)

print(f"Observed {k} sixes out of {n_rolls} rolls")
for die, p in zip(dice_names, posterior):
    print(f"P({die} | {k} sixes) = {p:.3f}")
print(f"True die: {true_die}")

# Likelihood plot
ks = np.arange(0, 11)
plt.figure(figsize=(8,5))
for die, p in dice_probs.items():
    plt.plot(ks, [binomial_prob(n_rolls, k, p) for k in ks], label=f"Die {die} (p={p})")
plt.xlabel('Number of sixes observed')
plt.ylabel('P(k | Die)')
plt.legend()
plt.title('Likelihoods of Observing k Sixes for Each Die')
plt.show()

# Accuracy evaluation
num_trials = 100
correct = 0
for _ in range(num_trials):
    true_die, k = simulate_round()
    posterior = posterior_given_k(k)
    predicted_die = dice_names[np.argmax(posterior)]
    correct += (predicted_die == true_die)

print(f"Accuracy over {num_trials} rounds: {correct/num_trials:.2f}")

# Posterior visualizations
posterior_matrix = np.array([posterior_given_k(k) for k in ks])
plt.figure(figsize=(7,5))
plt.imshow(posterior_matrix.T, cmap='viridis', aspect='auto')
plt.xticks(ks)
plt.yticks(range(3), dice_names)
plt.xlabel('Observed number of sixes (k)')
plt.ylabel('Posterior P(Die | k)')
plt.colorbar(label='Probability')
plt.title('Posterior Distribution over Dice for Different Observations')
plt.show()

## Problem B: Linear Regression
Given x=[-2,-1,0,1,2] and y=[7,4,3,4,7]. Fit a linear model using the normal equation.

In [ ]:
x = np.array([-2, -1, 0, 1, 2])
y = np.array([7, 4, 3, 4, 7])

# Construct design matrix (add intercept column)
X = np.c_[np.ones(len(x)), x]

# Normal equation: theta = (X^T X)^{-1} X^T y
theta = np.linalg.inv(X.T @ X) @ X.T @ y

# Predictions
y_pred = X @ theta

# Mean squared error
mse_linear = np.mean((y - y_pred)**2)

print('Linear theta:', theta, 'MSE:', mse_linear)

# Plot
plt.figure(figsize=(6,4))
plt.scatter(x, y, label='Data', color='blue')
plt.plot(x, y_pred, label='Linear fit', color='red')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.title('Linear Regression Fit')
plt.show()

## Problem C: Gradient Descent
Minimize f(w)=5(w−11)^4. Perform steps with α=1/400 and α=1/4000000. (Print the first 5 steps and visualize)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define function and gradient
def f(w):
    return 5 * (w - 11)**4

def grad_f(w):
    """Gradient of f(w) = 5(w-11)^4"""
    return 20 * (w - 11)**3

# Gradient Descent Function
def grad_descent_vals(w0, alpha, steps):
    """Run gradient descent and return history of w values"""
    w_history = [w0]
    w = w0
    for _ in range(steps):
        w = w - alpha * grad_f(w)
        w_history.append(w)
    return w_history

# Run for two learning rates
hist_1_400 = grad_descent_vals(13, 1/400, 200)
hist_1_4000000 = grad_descent_vals(13, 1/4000000, 200)

# Print first 5 steps for both learning rates
print("Learning rate α = 1/400:")
for i in range(min(5, len(hist_1_400))):
    print(f"  Step {i}: w = {hist_1_400[i]:.6f}, f(w) = {f(hist_1_400[i]):.6f}")

print("\nLearning rate α = 1/4000000:")
for i in range(min(5, len(hist_1_4000000))):
    print(f"  Step {i}: w = {hist_1_4000000[i]:.6f}, f(w) = {f(hist_1_4000000[i]):.6f}")

# Plot convergence
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(range(len(hist_1_400)), hist_1_400, marker='o', markersize=3, label='α=1/400')
plt.axhline(y=11, color='r', linestyle='--', label='Optimum (w=11)')
plt.xlabel('Iteration')
plt.ylabel('w')
plt.title('Convergence: α = 1/400 (larger step)')
plt.legend()
plt.grid()

plt.subplot(1, 2, 2)
plt.plot(range(len(hist_1_4000000)), hist_1_4000000, marker='o', markersize=3, label='α=1/4000000')
plt.axhline(y=11, color='r', linestyle='--', label='Optimum (w=11)')
plt.xlabel('Iteration')
plt.ylabel('w')
plt.title('Convergence: α = 1/4000000 (smaller step)')
plt.legend()
plt.grid()

plt.tight_layout()
plt.show()

# Also plot loss function value over iterations
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
losses_1_400 = [f(w) for w in hist_1_400]
plt.semilogy(range(len(losses_1_400)), losses_1_400, marker='o', markersize=3)
plt.xlabel('Iteration')
plt.ylabel('f(w) [log scale]')
plt.title('Loss over iterations: α = 1/400')
plt.grid()

plt.subplot(1, 2, 2)
losses_1_4000000 = [f(w) for w in hist_1_4000000]
plt.semilogy(range(len(losses_1_4000000)), losses_1_4000000, marker='o', markersize=3)
plt.xlabel('Iteration')
plt.ylabel('f(w) [log scale]')
plt.title('Loss over iterations: α = 1/4000000')
plt.grid()

plt.tight_layout()
plt.show()

ALL THE BEST!